In [ ]:
# Configure SCBFM_ROOT_DIR and optionally SCBFM_FIGURE_DIR before launching Jupyter.
from pathlib import Path
import os
import sys

_candidates = [Path(os.environ['SCBFM_REPO_DIR'])] if os.environ.get('SCBFM_REPO_DIR') else []
_candidates += [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'src' / 'main.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the checkout or set SCBFM_REPO_DIR.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from notebook_setup import ROOT_DIR, OUTPUT_DIR, FIGURE_DIR


In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import tables
import h5py
from scipy import sparse
from tqdm import tqdm
from pathlib import Path
import re

## Create gene list - intersection of all Ensembl ID sets from all datasets

In [ ]:
archs4 = ad.read_h5ad(str(ROOT_DIR / 'datasets/ARCHS4/archs4.h5ad'), backed = "r")
archs4_genes = set(archs4.var_names)
del archs4

In [ ]:
depmap = ad.read_h5ad(str(ROOT_DIR / 'datasets/DepMap/depmap.h5ad'), backed = "r")
depmap_genes = set(depmap.var_names)
del depmap

In [ ]:
disignatlas = ad.read_h5ad(str(ROOT_DIR / 'datasets/DiSignAtlas/disignatlas.h5ad'), backed = "r")
disignatlas_genes = set(disignatlas.var_names)
del disignatlas

In [ ]:
gdsc = ad.read_h5ad(str(ROOT_DIR / 'datasets/GDSC/gdsc.h5ad'), backed = "r")
gdsc_genes = set(gdsc.var_names)
del gdsc

In [ ]:
gtex = ad.read_h5ad(str(ROOT_DIR / 'datasets/GTEx/gtex.h5ad'), backed = "r")
gtex_genes = set(gtex.var_names)
del gtex

In [ ]:
survboard_data_dir = Path(str(ROOT_DIR / 'datasets/SurvBoard/data_reproduced'))
gene_info = pd.read_csv(str(REPO_ROOT / 'data/bulkformer_gene_info.csv'))

sym2ensg = dict(zip(gene_info["gene_symbol"].astype(str), gene_info["ensg_id"].astype(str)))
sym2ensg_upper = {k.upper(): v for k, v in sym2ensg.items()}

def map_gex_col_to_ensg(col):
    raw = col[len("gex_"):] if col.startswith("gex_") else col
    parts = [p.strip() for p in raw.split("|") if p.strip()]

    # Already Ensembl ID, with or without version: ENSG00000123456.7 -> ENSG00000123456
    for p in parts + [raw]:
        m = re.match(r"^(ENSG\d+)(?:\.\d+)?$", p)
        if m:
            return m.group(1)

    # Standard TCGA SurvBoard columns: gex_HUGO|ENTREZ
    symbol = parts[0] if parts else raw
    candidates = [
        symbol,
        symbol.upper(),
        symbol.replace(".", "-"),
        symbol.replace(".", "-").upper(),
    ]

    for s in candidates:
        if s in sym2ensg:
            return sym2ensg[s]
        if s in sym2ensg_upper:
            return sym2ensg_upper[s]

    return None

# TCGA-only SurvBoard expression files
survboard_files = sorted(
    (survboard_data_dir / "TCGA").glob("*_data_complete_modalities_preprocessed.csv")
)

if not survboard_files:
    raise FileNotFoundError(f"No TCGA SurvBoard files found under {survboard_data_dir / 'TCGA'}")

survboard_gene_sets = {}
mapping_rows = []

for path in survboard_files:
    project = path.parent.name
    cancer = path.name.replace("_data_complete_modalities_preprocessed.csv", "")

    header = pd.read_csv(path, nrows=0)
    gex_cols = [c for c in header.columns if c.startswith("gex_")]

    mapped = []
    unmapped = []
    seen = set()

    for col in gex_cols:
        ensg = map_gex_col_to_ensg(col)
        if ensg is None:
            unmapped.append(col)
            continue
        if ensg not in seen:
            mapped.append(ensg)
            seen.add(ensg)

    gene_set = set(mapped)
    survboard_gene_sets[(project, cancer)] = gene_set

    mapping_rows.append({
        "project": project,
        "cancer": cancer,
        "n_gex_cols": len(gex_cols),
        "n_mapped_unique_ensg": len(gene_set),
        "n_unmapped_or_duplicate": len(gex_cols) - len(gene_set),
        "n_unmapped_symbols": len(set(unmapped)),
        "example_unmapped": unmapped[:5],
    })

survboard_mapping_summary = pd.DataFrame(mapping_rows).sort_values(["project", "cancer"])
display(survboard_mapping_summary)

survboard_genes = set().union(*survboard_gene_sets.values())
survboard_genes_intersection = set.intersection(*survboard_gene_sets.values())

print("SurvBoard project: TCGA only")
print("SurvBoard files:", len(survboard_files))
print("SurvBoard mapped ENSG union:", len(survboard_genes))
print("SurvBoard mapped ENSG intersection:", len(survboard_genes_intersection))


In [ ]:
tcga = ad.read_h5ad(str(ROOT_DIR / 'datasets/TCGA/tcga.h5ad'), backed = "r")
tcga_genes = set(tcga.var_names)
del tcga

In [ ]:
print("ARCHS4 genes:", len(archs4_genes))
print("DepMap genes:", len(depmap_genes))
print("DisSignAtlas genes:", len(disignatlas_genes))
print("GDSC genes:", len(gdsc_genes))
print("GTEx genes:", len(gtex_genes))
print("TCGA genes:", len(tcga_genes))
print("SurvBoard genes (union):", len(survboard_genes))
print("SurvBoard genes (intersection):", len(survboard_genes_intersection))

In [ ]:
gene_list_ensemble_id = set(
    archs4_genes
    & depmap_genes
    & disignatlas_genes
    & gdsc_genes
    & gtex_genes
    & survboard_genes_intersection
    & tcga_genes
)


In [ ]:
len(gene_list_ensemble_id)

In [ ]:
genes = sorted(list(gene_list_ensemble_id))

with open(str(ROOT_DIR / 'datasets/gene_list.txt'), "w") as f:
    for g in genes:
        f.write(g + "\n")

In [ ]:
# query GENExCELL to see whether full gene_list is available there (internet access required, so run this cell separately if needed)

from pathlib import Path
import cellxgene_census

gene_list_path = Path(str(REPO_ROOT / 'data/gene_list.txt'))
census_version = "2025-11-08"
organism_key = "homo_sapiens"

with open(gene_list_path) as f:
    gene_list = [line.strip() for line in f if line.strip()]

with cellxgene_census.open_soma(census_version=census_version) as census:
    var = census["census_data"][organism_key].ms["RNA"].var.read(
        column_names=["soma_joinid", "feature_id", "feature_name"]
    ).concat().to_pandas()

available_feature_ids = set(var["feature_id"].astype(str))
available_feature_names = set(var["feature_name"].astype(str))

missing_by_ensg = sorted(set(gene_list) - available_feature_ids)
available_by_ensg = sorted(set(gene_list) & available_feature_ids)

print(f"Gene list size: {len(gene_list)}")
print(f"Available in CELLxGENE by Ensembl feature_id: {len(available_by_ensg)}")
print(f"Missing in CELLxGENE by Ensembl feature_id: {len(missing_by_ensg)}")

if missing_by_ensg:
    print("Missing genes:")
    for gene in missing_by_ensg:
        print(gene)
else:
    print("All genes from gene_list are available in CELLxGENE.")